# Compute Cornerness Score

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from src.utils import round_matrix

### Load blur derivative images

In [3]:
image_width = 210
image_height = 200

prefix = "assets/cornerness_score_expected_outputs/step3_blurred_"
left_ix_square = np.load(prefix + "left_ix_square.npy")
left_iy_square = np.load(prefix + "left_iy_square.npy")
left_ixiy = np.load(prefix + "left_ixiy.npy")
right_ix_square = np.load(prefix + "right_ix_square.npy")
right_iy_square = np.load(prefix + "right_iy_square.npy")
right_ixiy = np.load(prefix + "right_ixiy.npy")

## Manual Calculation

In [4]:
print("right_ix_square[10:13, 20:23]:")
print(right_ix_square[10:13, 20:23])

print("\nright_iy_square[10:13, 20:23]:")
print(right_iy_square[10:13, 20:23])

print("\nright_ixiy_square[10:13, 20:23]:")
print(right_ixiy[10:13, 20:23])

right_ix_square[10:13, 20:23]:
[[ 161.54882812  398.2109375   599.61621094]
 [ 400.71289062  902.51660156 1130.16503906]
 [ 951.59667969 1535.54882812 1400.99414062]]

right_iy_square[10:13, 20:23]:
[[1275.11523438  987.58984375  545.75683594]
 [ 691.17773438  584.69628906  406.47363281]
 [ 348.55371094  522.52148438  527.58789062]]

right_ixiy_square[10:13, 20:23]:
[[-351.04101562 -434.49023438 -329.93261719]
 [-132.77148438  -18.43652344  151.95800781]
 [ 369.12597656  733.81054688  761.2265625 ]]


In [5]:
prefix = "assets/corner_response/step4_"
expected_right_c = np.load(prefix + "right_corner_response.npy")

# np.set_printoptions(threshold=np.inf, linewidth=200)
print("expected_right_c[10:13, 20:23]:")
print(expected_right_c[10:13, 20:23])
print("\nexpected_right_c[11:21]:")
print(expected_right_c[11, 21])

Hx=right_ix_square[10:13, 20:23].sum()
Hy=right_iy_square[10:13, 20:23].sum()
Hxy=right_ixiy[10:13, 20:23].sum()

alpha = 0.04
cornerness_score = Hx * Hy - Hxy ** 2 - alpha * (Hx + Hy) ** 2
print("\ncornerness_score[10:13, 20:23]:")
print(cornerness_score)

expected_right_c[10:13, 20:23]:
[[       0. 19245117. 16565377.]
 [26783124. 36346256. 26842056.]
 [20390498. 21800401. 13669688.]]

expected_right_c[11:21]:
36346256.0

cornerness_score[10:13, 20:23]:
36346256.21550049


### Calculate cornerness scores functions

In [6]:
def compute_cornerness_score(
    ix_square, iy_square, ixiy, alpha, threshold, image_width, image_height
):
    top = 1000
    kernel_size = 3

    ext_ix_square = np.pad(ix_square, pad_width=1, mode="edge")
    ext_iy_square = np.pad(iy_square, pad_width=1, mode="edge")
    ext_ixiy = np.pad(ixiy, pad_width=1, mode="edge")

    corners = []
    for i in range(image_height):
        for j in range(image_width):
            H_ix = ext_ix_square[i : i + kernel_size, j : j + kernel_size].sum()
            H_iy = ext_iy_square[i : i + kernel_size, j : j + kernel_size].sum()
            H_ixiy = ext_ixiy[i : i + kernel_size, j : j + kernel_size].sum()
            
            # C = det(H) - alpha * (trace(H))^2
            det_H = H_ix * H_iy - H_ixiy**2
            trace_H = H_ix + H_iy
            score = det_H - alpha * (trace_H**2)

            if score > threshold:
                corners.append((i, j, score))
    corners.sort(key=lambda x: x[2], reverse=True)
    return corners[:top]

In [7]:
alpha = 0.04
threshold = 10 ** 6

cornerness_score_left = compute_cornerness_score(left_ix_square, left_iy_square, left_ixiy, 
                                                 alpha, threshold, image_width, image_height)
cornerness_score_right = compute_cornerness_score(right_ix_square, right_iy_square, right_ixiy, 
                                                  alpha, threshold, image_width, image_height)